# Rulebook Inspection

Standalone inspection tools — read-only, do not modify any pipeline data.

- **Section 1:** Check which expected categories are present or missing in the active rulebook
- **Section 2:** Read and print AI mapping logs (prompt + model response) for a given file

In [ ]:
import json
import sys
from pathlib import Path

# Resolve sys.path to the t12_pipeline src/ so the package imports work
# regardless of where the notebook is launched from.
_HERE = Path.cwd().resolve()
_SRC = next(
    (p for p in [_HERE, *_HERE.parents] if (p / 't12_pipeline' / 'config' / 'paths.py').is_file()),
    None,
)
if _SRC and str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from t12_pipeline.config.paths import (
    DB_RULEBOOKS_DIR,
    MAPPED_T12_DOCS_DIR,
    OUTPUT_DOCS_BASE_DIR,
)
from t12_pipeline.config.rulebook_utils import get_rulebook_categories

# The active rulebook is the newest mapping_rulebook_*.json in db/rulebooks/.
# Same logic both the pipeline and the UI bridge use to pick the rulebook.
_candidates = sorted(DB_RULEBOOKS_DIR.glob('mapping_rulebook_*.json'))
if not _candidates:
    raise FileNotFoundError(
        f"No rulebook in {DB_RULEBOOKS_DIR}. "
        "Run `python -m t12_pipeline.build_rulebook.run` first."
    )
LATEST_RULEBOOK_PATH = _candidates[-1]

print(f'Latest rulebook: {LATEST_RULEBOOK_PATH.name}')
print(f'Mapped docs dir: {MAPPED_T12_DOCS_DIR}')

## Section 1 — Category Coverage Check

Checks which expected canonical categories are present or missing in the active v2 rulebook.
Update `expected_keys` below if the chart of accounts changes.

In [ ]:
files = sorted(MAPPED_T12_DOCS_DIR.glob('*.xlsm'))
files = [f for f in files if not f.name.startswith('~$')]
print(f'xlsm source files ({len(files)} total):')
for f in files:
    print(f'  {f.name}')

In [ ]:
expected_keys = [
    '(Loss) Gain To Lease', 'Gross Potential Rent', 'Rental Concessions',
    'Vacancy Loss', 'Non Revenue Units', 'Bad Debt',
    'Utility Rebill - Electric', 'Utility Rebill - Gas', 'Utility Rebill - Water/Sewer',
    'Utility Rebill - Trash Removal', 'Utility Rebill - Cable, Internet, Other',
    'Utility Rebill - Other', 'Other Income Ops', 'Parking Income',
    'Laundry And Vending', 'Other Income: Value Add',
    'Repairs And Maintenance', 'Turnover Expenses', 'Contracted Services',
    'Professional Fees', 'Payroll Salaries/Wages', 'Payroll Burden',
    'Utilities Electric', 'Utilities Gas', 'Utilities Water/Sewer',
    'Utilities Trash Removal', 'Utilities Cable, Internet, Other', 'Utilities Other',
    'Administrative', 'Marketing And Retention', 'Management Fee',
    'Real Estate Taxes', 'Insurance',
    'Capex Ongoing', 'Capex Rehab / Replace', 'Capex Major',
]

with open(LATEST_RULEBOOK_PATH, 'r') as f:
    rulebook = json.load(f)

# Current rulebook shape: { certain, contextual, no_conflict, unresolved }
# get_rulebook_categories pulls the union of canonical categories actually used.
categories = sorted(get_rulebook_categories(rulebook))
categories_lower = {c.strip().lower() for c in categories}

present = [k for k in expected_keys if k.strip().lower() in categories_lower]
missing = [k for k in expected_keys if k.strip().lower() not in categories_lower]

print(f'Rulebook source: {LATEST_RULEBOOK_PATH.name}')
print(f'  Buckets:           certain={len(rulebook.get("certain", {}))}, '
      f'contextual={len(rulebook.get("contextual", {}))}, '
      f'no_conflict={len(rulebook.get("no_conflict", {}))}, '
      f'unresolved={len(rulebook.get("unresolved", {}))}')
print(f'  Unique categories: {len(categories)}')
print()
print(f'Expected: {len(expected_keys)} / Present: {len(present)} / Missing: {len(missing)}')

if missing:
    print('\nMissing keys:')
    for m in missing:
        print(f'  - {m}')
else:
    print('\nAll expected categories are present.')

print('\nAll categories in rulebook:')
for k in categories:
    print(f'  {k}')

## Section 2 — AI Mapping Log Reader

Reads and prints the full LLM prompt and raw response from an AI mapping log file.
Useful for inspecting what context the model received and how it responded.

Change `log_file` below to inspect a different property.

In [ ]:
# The setup cell at the top already imported what we need
# (OUTPUT_DOCS_BASE_DIR). Section 2 reuses that scope.

In [ ]:
# Walk every past pipeline run and list AI-mapping log files.
# AI_MAPPING_LOGS_DIR in paths.py points to the CURRENT timestamp only — but we
# want to inspect any past run, so glob through output_docs/<ts>/2_mapping/ai_mapping_logs.
log_dirs = sorted(OUTPUT_DOCS_BASE_DIR.glob('*/2_mapping/ai_mapping_logs'))

if not log_dirs:
    print(f'No AI mapping logs found yet under {OUTPUT_DOCS_BASE_DIR}/*/2_mapping/ai_mapping_logs/')
    print('Run the pipeline first: python -m t12_pipeline.pipeline.run_pipeline --stages 0,1,2')
else:
    print(f'Found {len(log_dirs)} run(s) with AI mapping logs:\n')
    for d in log_dirs:
        run_id = d.parent.parent.name
        log_files = sorted(d.glob('*.json'))
        print(f'  {run_id}  ({len(log_files)} log files)')
        for f in log_files:
            print(f'    {f.name}')

In [ ]:
# Default: pick the most recent run's first log file.
# To inspect a different one, copy/paste a path from the cell above and set
# `log_file` explicitly, e.g.:
#   log_file = OUTPUT_DOCS_BASE_DIR / '2026_06_07_0510' / '2_mapping' / 'ai_mapping_logs' / 'Brownsburg Crossing T12 05-25_logs.json'
if not log_dirs:
    raise FileNotFoundError('No AI mapping logs available — run the pipeline first.')

latest_dir = log_dirs[-1]
candidates = sorted(latest_dir.glob('*.json'))
if not candidates:
    raise FileNotFoundError(f'No .json log files in {latest_dir}')

log_file = candidates[0]

with open(log_file, 'r') as f:
    log = json.load(f)

print(f'Log file: {log_file.relative_to(OUTPUT_DOCS_BASE_DIR)}')
print(f'Entries: {len(log)}')

In [9]:
# Print the first entry — change index to inspect other rows
for entry in log:
# entry = log[0]
    print('=== PROMPT ===')
    print(entry.get('llm_final_prompt'))
    
    print('\n=== RAW OUTPUT ===')
    print(json.dumps(entry.get('llm_raw_output'), indent=2))

=== PROMPT ===
You are an expert real estate financial accountant operating as an autonomous AI mapping assistant within the 'Shivam Excel AI Automation' pipeline. 
Your specific role in this larger workflow is to assist when our primary rulebook fails to automatically map a line item. When an unrecognized item from a property's T12 (Trailing 12-month) financial statement appears, your job is to analyze the item and suggest a logical mapping. 

Your suggestions help us track these unrecognized items, making it much easier for a human operator to manually review and override mappings later in the workflow.

### System & Data Context
To help you make an accurate decision, you are provided with a JSON object (`item_to_map`) that contains the following extracted data:
- `file_name`: The source Excel file, providing context on the specific property.
- `section_context`: The nearest header or section title found directly above the item in the original Excel file (e.g., "ADMINISTRATIVE EXPENS